# Project Parallax — Phase 1: Data Validation

**Status:** Skeleton — Phase 1 in progress  
**Purpose:** Assess coverage quality of the research universe, quantify survivorship bias exposure, and determine whether the data strategy is sufficient to support Phases 2–7.

This notebook is a working document. Results and conclusions will be filled in as Phase 1 proceeds.

---

## Phase 1 Research Questions

1. What fraction of the S&P 500's historical constituent universe can be retrieved from the free data source (yfinance)?
2. How large is the survivorship bias exposure — what share of the historical panel consists exclusively of companies that survived to current membership?
3. Is the survivorship bias exposure large enough to materially distort dispersion, correlation, or attribution estimates?
4. Is the free data source sufficient for the full research scope, or is escalation to a paid source (Sharadar/Nasdaq Data Link) required?

## Decision Threshold (D-001)

If survivorship exposure exceeds ~15% of the historical panel, escalate the data strategy. If the free source cannot cover at least 80% of historical S&P 500 constituents for the core study period, treat free-source results as preliminary estimates only.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yfinance as yf
import warnings

warnings.filterwarnings('ignore')

# Project source
import sys
sys.path.insert(0, '..')

from src.validation import UniverseCoverage
# Note: assess_coverage, flag_survivorship_risk, generate_bias_report are Phase 1 stubs.
# This notebook will drive the implementation of those functions.

print('Imports OK')

---

## Section 1: Universe Definition

### 1.1 Current S&P 500 Constituents

Current constituents are the starting point. They are explicitly **not** equivalent to historical point-in-time constituents — that gap is what this notebook measures.

In [ ]:
# ── Current S&P 500 constituent list ──────────────────────────────────────────
# Source: Wikipedia scrape (snapshot, not point-in-time).
# This is the CURRENT universe only. Historical universe is handled in Section 2.

sp500_table = pd.read_html(
    'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
)[0]

current_tickers = sp500_table['Symbol'].str.replace('.', '-', regex=False).tolist()
current_sectors = sp500_table.set_index('Symbol')['GICS Sector']

print(f'Current S&P 500 constituents: {len(current_tickers)}')
print(f'\nSector distribution:')
print(current_sectors.value_counts())

### 1.2 Study Period

The study period is an open design decision (D-002). Candidate ranges are documented here; the final selection will be registered in RESEARCH_LOG.md.

In [ ]:
# ── Study period candidates ────────────────────────────────────────────────────
# These are candidates only. Selection pending Phase 1 data feasibility results.

STUDY_PERIOD_CANDIDATES = {
    'short':    ('2010-01-01', '2024-12-31'),   # 15 years
    'medium':   ('2005-01-01', '2024-12-31'),   # 20 years
    'extended': ('2000-01-01', '2024-12-31'),   # 25 years
}

# Working assumption for this notebook — update after Phase 1 results:
STUDY_START = '2010-01-01'
STUDY_END   = '2024-12-31'

print(f'Working study period: {STUDY_START} to {STUDY_END}')
print(f'(Open design decision D-002 — subject to Phase 1 results)')

---

## Section 2: Historical Universe — Survivorship Bias Exposure

### 2.1 Point-in-Time Constituent History

The S&P 500's historical constituent changes are publicly available through secondary sources. This section retrieves what is available and documents the gap where it is not.

In [ ]:
# ── Historical constituent changes ─────────────────────────────────────────────
# Source: Wikipedia S&P 500 changes table (secondary, not authoritative).
# Authoritative historical data requires a paid source (e.g., Compustat, CRSP).
# This section documents what is available for free and what is not.

# TODO (Phase 1): Retrieve and parse the Wikipedia S&P 500 changes table.
# Reconstruct point-in-time constituent lists at each year-end.
# Document: how many additions and removals per year, and the tickers affected.

# PLACEHOLDER — replace with actual implementation:
historical_changes = None  # pd.DataFrame of (date, ticker, action: 'added'/'removed')

print('Historical constituent data: NOT YET RETRIEVED (Phase 1 TODO)')
print('\nThis is the primary survivorship bias source to quantify.')

In [ ]:
# ── Survivorship exposure estimate ─────────────────────────────────────────────
# Without point-in-time data, we estimate survivorship exposure as:
#   (current constituents who have been in the index the entire study period)
#   / (total constituents who were in the index at any point during the study period)
#
# This requires historical change data above. Until it is retrieved, we use
# a theoretical lower bound from literature:
#   Over a 10-year window, roughly 30-40% of S&P 500 constituents turn over.
#   Using only current constituents therefore excludes ~30-40% of the historical panel.

# TODO (Phase 1): Replace with computed value once historical data is available.
SURVIVORSHIP_EXPOSURE_ESTIMATE = None

print('Survivorship exposure: NOT YET COMPUTED')
print('Theoretical range (literature): 0.30 to 0.40 over a 10-year window')
print('\nIf computed exposure exceeds 0.15, escalate data strategy per D-001.')

---

## Section 3: Data Retrieval — Coverage Assessment

### 3.1 Price and Return Retrieval

This section retrieves monthly returns for current S&P 500 constituents and measures coverage completeness. Coverage here is for the **current** universe only — the survivorship-biased sample. The historical gap is documented in Section 2.

In [ ]:
# ── Retrieve monthly price data ────────────────────────────────────────────────
# Using yfinance for free-tier retrieval.
# NOTE: yfinance uses current tickers. Delisted securities are not available.
# This is a known limitation — documented here, not treated as a cleanup task.

# TODO (Phase 1): Run this retrieval, document results in RESEARCH_LOG.md.

# Sample retrieval (small subset first to validate the pipeline):
SAMPLE_TICKERS = current_tickers[:20]  # First 20 for validation

print(f'Sample retrieval: {len(SAMPLE_TICKERS)} tickers')
print(f'Period: {STUDY_START} to {STUDY_END}')
print('\nRun the cell below to execute retrieval.')

In [ ]:
# ── Execute sample retrieval ───────────────────────────────────────────────────
# Adjust interval and tickers as Phase 1 progresses.

raw_prices = yf.download(
    tickers=SAMPLE_TICKERS,
    start=STUDY_START,
    end=STUDY_END,
    interval='1mo',
    auto_adjust=True,
    progress=False,
)['Close']

print(f'Retrieved: {raw_prices.shape[0]} months × {raw_prices.shape[1]} securities')
print(f'Date range: {raw_prices.index[0].date()} to {raw_prices.index[-1].date()}')
print(f'\nMissing data summary (% of periods with NaN per ticker):')
missing_pct = raw_prices.isna().mean().sort_values(ascending=False)
print(missing_pct.head(10))

In [ ]:
# ── Compute monthly returns ────────────────────────────────────────────────────

monthly_returns = raw_prices.pct_change()

# Drop the first row (NaN from pct_change) and any rows that are all-NaN
monthly_returns = monthly_returns.iloc[1:].dropna(how='all')

print(f'Return panel: {monthly_returns.shape[0]} months × {monthly_returns.shape[1]} securities')
print(f'\nFirst 5 rows:')
monthly_returns.head()

### 3.2 Coverage Quality Metrics

These metrics characterize the quality of the retrieved panel. They drive the data strategy decision (D-001).

In [ ]:
# ── Coverage quality assessment ────────────────────────────────────────────────
# TODO (Phase 1): Call src.validation.assess_coverage() once implemented.
# For now, compute equivalent statistics directly.

n_periods = len(monthly_returns)
n_securities = len(monthly_returns.columns)

# Securities with full coverage (no missing values across the full panel)
full_coverage_mask = monthly_returns.notna().all()
n_full = full_coverage_mask.sum()

# Securities with partial coverage (some missing values)
partial_coverage_mask = monthly_returns.notna().any() & ~full_coverage_mask
n_partial = partial_coverage_mask.sum()

# Securities with no data at all
no_coverage_mask = ~monthly_returns.notna().any()
n_missing = no_coverage_mask.sum()

print('Coverage Summary (sample universe)')
print('=' * 40)
print(f'Total securities:          {n_securities}')
print(f'Full coverage:             {n_full} ({n_full/n_securities:.1%})')
print(f'Partial coverage:          {n_partial} ({n_partial/n_securities:.1%})')
print(f'No data:                   {n_missing} ({n_missing/n_securities:.1%})')
print(f'Study period length:       {n_periods} months')

In [ ]:
# ── Coverage timeline ──────────────────────────────────────────────────────────
# How many securities have data in each month? Drops indicate delisting events
# or data gaps.

coverage_by_month = monthly_returns.notna().sum(axis=1)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(coverage_by_month.index, coverage_by_month.values, linewidth=1.2, color='steelblue')
ax.set_xlabel('Date')
ax.set_ylabel('Securities with data')
ax.set_title('Coverage Timeline — Securities with Return Data by Month')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/figures/phase1_coverage_timeline.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: docs/figures/phase1_coverage_timeline.png')

---

## Section 4: Factor Data Retrieval

FF6 factor data from the Kenneth French Data Library is required for Phase 4 (Factor Attribution Engine). This section validates that the factor data can be retrieved and aligned to the security return panel.

In [ ]:
# ── FF6 factor data validation ─────────────────────────────────────────────────
# Source: Kenneth French Data Library
# URL: https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html
#
# This section validates retrieval feasibility only.
# Full implementation deferred to Phase 4 (load_ff6_factors stub in factors.py).

# NOTE: The pandas_datareader library can retrieve French data directly.
# Validate that it is available.

try:
    import pandas_datareader.data as web
    print('pandas_datareader: available')
except ImportError:
    print('pandas_datareader: NOT installed — run: pip install pandas-datareader')
    print('French factor data will need to be downloaded manually until then.')

In [ ]:
# ── Retrieve FF6 monthly factors (if pandas_datareader available) ──────────────
# Retrieves: Mkt-RF, SMB, HML, RMW, CMA, Mom
# Data is in percentage form — divide by 100 for decimal returns.

# TODO (Phase 1): Run retrieval, validate date alignment with security returns.

try:
    ff5 = web.DataReader('F-F_Research_Data_5_Factors_2x3', 'famafrench',
                         start=STUDY_START, end=STUDY_END)[0]
    mom = web.DataReader('F-F_Momentum_Factor', 'famafrench',
                         start=STUDY_START, end=STUDY_END)[0]

    ff6 = ff5.join(mom)
    ff6.columns = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF', 'Mom']
    ff6 = ff6 / 100.0  # Convert to decimal

    print(f'FF6 factors retrieved: {ff6.shape[0]} months')
    print(f'Date range: {ff6.index[0]} to {ff6.index[-1]}')
    print(f'Columns: {list(ff6.columns)}')
    ff6.tail()

except Exception as e:
    print(f'Factor retrieval failed: {e}')
    print('Manual download from French Data Library required.')
    ff6 = None

In [ ]:
# ── Date alignment check ───────────────────────────────────────────────────────
# Verify that factor dates align with security return dates.
# Misalignment here would produce incorrect factor exposures.

if ff6 is not None and len(monthly_returns) > 0:
    # French data uses period-end month index; yfinance uses the first trading day.
    # Alignment requires normalization to a common frequency.
    security_months = monthly_returns.index.to_period('M')
    factor_months = ff6.index.to_period('M') if hasattr(ff6.index, 'to_period') else pd.PeriodIndex(ff6.index, freq='M')

    common_months = set(security_months) & set(factor_months)
    print(f'Security return months:  {len(security_months)}')
    print(f'Factor return months:    {len(factor_months)}')
    print(f'Common months:           {len(common_months)}')

    if len(common_months) < len(security_months) * 0.95:
        print('\nWARNING: Less than 95% date overlap. Investigate alignment before proceeding.')
    else:
        print('\nDate alignment: OK')
else:
    print('Alignment check skipped — data not available.')

---

## Section 5: Data Strategy Decision

This section synthesizes Phase 1 findings into a data strategy recommendation. The outcome will be registered in RESEARCH_LOG.md.

In [ ]:
# ── Data strategy assessment ───────────────────────────────────────────────────
# Fill in once Phase 1 retrieval and survivorship analysis is complete.

# Decision criteria (from PROJECT_CHARTER.md D-001 through D-003):

ASSESSMENT = {
    'coverage_pct': None,             # Fill: % of current constituents with full history
    'survivorship_exposure': None,    # Fill: estimated % of panel that is survivorship-biased
    'factor_data_available': None,    # Fill: True/False
    'date_alignment_ok': None,        # Fill: True/False
    'recommendation': None,           # Fill: 'proceed_free' / 'proceed_with_caveats' / 'escalate'
    'escalation_trigger_met': None,   # Fill: True/False (survivorship > 0.15)
}

print('Data Strategy Assessment — Phase 1')
print('=' * 40)
for k, v in ASSESSMENT.items():
    status = str(v) if v is not None else 'PENDING'
    print(f'{k:35s}: {status}')

print('\nThis assessment will be registered in RESEARCH_LOG.md once complete.')

---

## Section 6: Phase 1 Deliverables Checklist

From PROJECT_CHARTER.md Section 9 (Phase 1):

| Deliverable | Status |
|---|---|
| D1: Universe coverage summary | ◻ Pending |
| D2: Survivorship bias quantification | ◻ Pending |
| D3: Survivorship / point-in-time bias diagnostic report | ◻ Pending |
| D4: Data strategy decision (proceed / escalate) | ◻ Pending |
| D5: Decision registered in RESEARCH_LOG.md | ◻ Pending |

---

*Notebook created: 2026-08-18. Phase 1 data work in progress.*